
# 練習問題: 命令レベル並列性 (ILP) --- ベクトル長 nl を最適化する

## 目標

1スレッドのまま, ベクトル長 `nl` (同時に進める独立な計算の本数) を増やすと, **命令レベル並列性 (ILP)** が引き出されて性能が上がることを体験する. これはGPUが1コアで多数のスレッドを走らせて性能を出すのと本質的に同じ理屈 (演算の遅延を, 独立な計算で埋める) である.

これはマルチコア化 (`00_simd_multicore`) の前段にあたる, **1コアの性能を引き出す**ための問題である.

## 課題

`tune_nl.cpp` は, 互いに独立な漸化式 `t = 0.99*t + 1` を `nl` 本まとめてベクトル型で進める (1スレッド).
プログラムの先頭に

```cpp
enum { nl = 8 };   /* ★ ベクトル長 */
```

がある. この `nl` を **1, 2, 4, 8, 16, 32 (2のべき乗)** と変えて毎回コンパイルし直し, GFLOPS がどう変化するかを調べよ. `m` (漸化式の本数) は `nl` の倍数にすること (引数で `nl` 以上の倍数を与えるとよい).

```
nvc++ -fast tune_nl.cpp -o tune_nl.exe
./tune_nl.exe 64 $((100 * 1000 * 1000))
```

`nl` を増やすたびにコンパイルし直す:

```
# enum { nl = 16 }; に書き換えてから
nvc++ -fast tune_nl.cpp -o tune_nl.exe && ./tune_nl.exe 64 $((100 * 1000 * 1000))
```

### Fortran

ベクトル型は C/C++ 独自の拡張なので, `tune_nl.f90` では `nl` 本まとめる書き方はできない. 代わりに `m` 本の漸化式を素直なループで書き, コンパイラの自動最適化に任せている. C++ で `nl` を上げたときの性能と, Fortran 版の性能を比べてみよ.

## 期待される結果・考察

- `nl=1` では, 1本の漸化式の `t = 0.99*t + 1` を待ちながら進めるため, 演算器が遊んで GFLOPS は低い.
- `nl` を増やすと, 独立な計算が同時に流れて演算器が埋まり, GFLOPS が上がる. ある `nl` で 1コアの限界に達し, それ以上は頭打ち (またはレジスタ不足で低下) する.
- 最良の `nl` を見つけよ. そのうえで `00_simd_multicore` のようにマルチコア (`#pragma omp parallel for`) と組み合わせると, さらに台数効果が乗る.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ tune_nl.cpp
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

enum { nl = 8 };   /* ★ ベクトル長. 1,2,4,8,16,32 と変えて再コンパイルし最良を探す */
typedef double doublev __attribute__((vector_size(sizeof(double) * nl)));

static double sec() {
  struct timeval t; gettimeofday(&t, 0);
  return t.tv_sec + t.tv_usec * 1e-6;
}

int main(int argc, char ** argv) {
  /* m 本の独立な漸化式 t = 0.99*t + 1 を, 各 n 回進める. m は nl の倍数. */
  long m = (1 < argc ? atol(argv[1]) : nl);
  long n = (2 < argc ? atol(argv[2]) : 100L * 1000 * 1000);
  double s = 0.0;
  double t0 = sec();
  for (long g = 0; g < m; g += nl) {
    doublev t;
    for (int k = 0; k < nl; k++) t[k] = 1.0;
    /* nl 本を1命令でまとめて進める (命令レベル並列) */
    for (long j = 0; j < n; j++) t = 0.99 * t + 1.0;
    for (int k = 0; k < nl; k++) s += t[k];
  }
  double dt = sec() - t0;
  double flops = 2.0 * (double)m * (double)n;
  printf("nl=%d, m=%ld, n=%ld : %.3f GFLOPS (s=%f)\n", nl, m, n, flops / dt * 1e-9, s);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore tune_nl.cpp -o tune_nl_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./tune_nl_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ tune_nl.f90
! 注: C/C++ のベクトル型 (vector_size) による「nl 本を明示的にまとめて進める」技法は
! Fortran には無い. ここでは m 本の独立な漸化式を素直なループで書き, コンパイラの
! 自動ベクトル化・命令レベル並列に任せる (どこまで速くなるかは nvfortran 次第).
program tune_nl
  character(len=32) :: arg
  integer(8) :: m, n, i, j, c0, c1, rate
  real(8) :: t, s, dt, flops
  m = 8
  n = 100_8 * 1000 * 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) m
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) n
  end if
  s = 0.0d0
  call system_clock(c0, rate)
  do i = 1, m
     t = 1.0d0
     do j = 1, n
        t = 0.99d0 * t + 1.0d0
     end do
     s = s + t
  end do
  call system_clock(c1)
  dt = real(c1 - c0, 8) / real(rate, 8)
  flops = 2.0d0 * real(m, 8) * real(n, 8)
  print "(a,i0,a,i0,a,f0.3,a,f0.6,a)", &
       "m=", m, ", n=", n, " : ", flops / dt * 1e-9, " GFLOPS (s=", s, ")"
end program tune_nl

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore tune_nl.f90 -o tune_nl_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./tune_nl_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:tune_nl.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:tune_nl.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:tune_nl.cpp}